## 2026 EY AI & Data Challenge - Landsat Data Extraction Notebook

This notebook demonstrates Landsat data extraction and the creation of an output file to be used by the benchmark notebook. The baseline data is [Landsat Collection 2 Level 2](https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2) data from the MS Planetary Computer catalog.

**Caution**... This notebook requires significant execution time as there are 9,319 data points (unique locations and times) used for data extraction from the Landsat archive. The code takes about 7 hours to run to completion on a typical laptop computer with a typical internet connection. Lower execution times are likely possible with optimization of the data extraction process and the use of cloud computing services.


### Load In Dependencies
The following code installs the required Python libraries (found in the requirements.txt file) in the Snowflake environment to allow successful execution of the remaining notebook code. After running this code for the first time, it is required to “restart” the kernal so the Python libraries are available in the environment. This is done by selecting the “Connected” menu above the notebook (next to “Run all”) and selecting the “restart kernal” link. Subsequent runs of the notebook do not require this “restart” process. 

In [ ]:
!pip install uv
!uv pip install  -r ../../requirements.txt

In [1]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

from pathlib import Path

import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from datetime import date, timezone
from tqdm import tqdm
import os

### Extracting Landsat Data Using API Calls

The API-based method allows us to efficiently access **Landsat** data for specific coordinates and time periods, ensuring scalability and reproducibility of the process.

Through the API, we can query individual bands or compute indices like **NDMI** on the fly. This approach reduces storage requirements and simplifies data preprocessing, making it ideal for large-scale environmental and water quality analysis.

The **compute_Landsat_values** function extracts Landsat surface reflectance values for specific sampling locations using a 100 m focal buffer around each point. For each location:

- A bounding box (bbox) is created around the latitude and longitude coordinates.
- The Microsoft Planetary Computer API is queried for Landsat-8 Level-2 surface reflectance imagery within the date range.
- The nearest low-cloud (<10% cloud cover) scene is selected, and the specified bands (**green**, **nir08**, **swir16**, **swir22**) are loaded.
- Median values of the pixels within the bounding box are computed to reduce the effect of noise or outliers.

**Why the buffer value is 0.00089831**

We want a ~100 m buffer around each point.  
At the equator, 1 degree ≈ 110 km.

Therefore, the degree equivalent of 100 m is:

*buffer_deg ≈ 100 m / 110,000 m per degree ≈ 0.00089831*

This value ensures that the buffer approximately matches the pixel resolution of Landsat imagery, capturing a ~100 m area around each sampling location.


#### Extension

Since API calls are so expensive, I've opted to do one bulk load for features to store the CSV files locally. Then, faster iterative development can be done based on that dataset

In [2]:
# Setup
tqdm.pandas()

def compute_Landsat_values(row):
    lat = row['Latitude']
    lon = row['Longitude']
    date = pd.to_datetime(row['Sample Date'], dayfirst=True, errors='coerce')

    # Buffer size for ~100m 
    bbox_size = 0.00089831  
    bbox = [
        lon - bbox_size / 2,
        lat - bbox_size / 2,
        lon + bbox_size / 2,
        lat + bbox_size / 2
    ]

    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )

    # Wider search range, we'll filter to nearest date later

    # systematically set this search such that all features from different landsat missions can be captured


    # actually just include both, since theres a chance landsat-8 doesn't capture a cloud thimg
    # just make sure that landsat 8 is chosen amap and only if it has a valid cloud filter
    # we can just partition here since we want to prioritise landsat8 readings for more features (the coastal feature)
    search = catalog.search(
        collections=["landsat-c2-l2"],
        bbox=bbox,
        datetime="2011-01-01/2015-12-31",   # landsat 8 operationally starts in April 2013, overlap just incase
        query={
            "platform": {"in": ["landsat-4", "landsat-5", "landsat-7", "landsat-8"]},
            "eo:cloud_cover": {"lt": 10},
        },
    )

    # this effectively returns a list of items, I imagine I would do a seperate search for each possible
    # landsat mission, zip with info on which landsat mission it is, then aggregate all lists together
    # (or "agrregate" through just successive for loops, depending on )

    # actually I don't even need to do that, just label each items as items_landsatX, then sort within each instance
    # (since I imagine landsat missions don't overlap) append all instances together
    # landsat missions overlap, so don't sort individually since you need to sort everything anyways

    items = search.item_collection()

    if not items:
        raise ValueError("Missing Items")
        # return pd.Series({
            # "has_coastal": np.nan, "coastal": np.nan, "nir": np.nan, "red": np.nan, "blue": np.nan,
            # "green": np.nan, "swir16": np.nan, "swir22": np.nan
        # })

    # weigh to prefer landsat 8 AMAP, effectively number of days to change
    # (i.e. if nothing in landsat-7 for 5 days, move to landsat-4)
    PLATFORM_WEIGHTS = {
        "landsat-8": 0,
        "landsat-7": 5,
        "landsat-4": 8,
        "landsat-5": 8,
    }
    
    def platform_priority(item, sample_date_utc):
        item_date = item.datetime.replace(tzinfo=timezone.utc)
        time_diff = abs(item_date - sample_date_utc)
        platform = item.properties["platform"]
    
        platform_penalty = PLATFORM_WEIGHTS.get(platform, 999)
        
        return time_diff.days + platform_penalty

    try:
        # Convert sample date to UTC
        sample_date_utc = date.tz_localize("UTC") if date.tzinfo is None else date.tz_convert("UTC")

        # Pick the item closest to the sample date
        # originally sorted? Why? I don't know but min is more efficient
        item = min(
            items,
            key=lambda x: platform_priority(x, sample_date_utc)
        )
        selected_item = pc.sign(item)

        # Load required bands
        # landsat8 has coastal
        if item.properties["platform"] == "landsat-8":
            bands_of_interest = ["coastal", "red", "blue", "green", "nir08", "swir16", "swir22"]
        else:
            bands_of_interest = ["red", "blue", "green", "nir08", "swir16", "swir22"]
            
        data = stac_load([selected_item], bands=bands_of_interest, bbox=bbox).isel(time=0)
        
        # Compute medians
        def get_clean_median(band_name):
            if band_name not in data: 
                if band_name == "coastal":
                    return np.nan   # if coastal missed, we mark with np.nan, do post processing
                                    # depending on what model you want, for now keep the sentinel with this

                raise ValueError("band_name other than coastal missing")
            
            band_data = data[band_name].where(data[band_name]!= 0)
            # Apply standard Landsat C2 L2 scaling formula
            val = float(band_data.median(skipna=True).values)
            return (val * 0.0000275 - 0.2) if not np.isnan(val) else np.nan

        return pd.Series({
            "has_coastal": item.properties["platform"] == "landsat-8",
            "coastal": get_clean_median("coastal"),
            "red":     get_clean_median("red"),
            "blue":    get_clean_median("blue"),
            "green":   get_clean_median("green"),
            "nir":     get_clean_median("nir08"),
            "swir16":  get_clean_median("swir16"),
            "swir22":  get_clean_median("swir22"),
        })
    
    except Exception as e:
        print(e)
        raise

        # return pd.Series({
            # "has_coastal": has_coastal, "coastal": np.nan, "nir": np.nan, "red": np.nan, "blue": np.nan,
            # "green": np.nan, "swir16": np.nan, "swir22": np.nan
        # })

### Extracting features for the training dataset

In [3]:
Water_Quality_df=pd.read_csv('../../data/raw/water_quality_training_dataset.csv')   # move data directory
display(Water_Quality_df.head())

In [4]:
Water_Quality_df.shape

### Note

The Landsat data extraction process for all 9,319 locations typically requires more than 7 hours when executed in a single run. During long executions, you may occasionally encounter API limits, timeout errors, or request failures. To avoid these interruptions, we recommend running the extraction in smaller batches.

In this notebook, we provide a sample code snippet demonstrating how to extract data for the first 200 locations. Participants are encouraged to follow the same batching approach to extract data for all 9,319 locations safely and efficiently.

We have already executed the full extraction for all 9,319 locations and saved the output to **landsat_features_training.csv**, which will be used in the benchmark notebook.  
Similarly, participants can extract Landsat features in batches, combine the batch outputs, and save the final merged dataset as **landsat_features_training.csv** to ensure the benchmark notebook runs smoothly.


### Test for one batch

In [ ]:
Water_Quality_df_200 = Water_Quality_df.tail(200)
Water_Quality_df_200.shape

# Extract band values from Landsat for training dataset
train_features_path = "../../data/landsat_features_training_200.csv"

print("Running Landsat feature extraction for training data...")
landsat_train_features = Water_Quality_df_200.progress_apply(compute_Landsat_values, axis=1)
landsat_train_features.to_csv(train_features_path, index=False)

In [ ]:
landsat_train_features.head(5)

#### Since we know that works, lets write a script that does it for the entire 10k~ rows

In [ ]:
# Define the relative path
file_path = Path("../../data/raw/landsat_all_features_training.csv")

# Check if it exists
if file_path.exists():
    print(f"File found: {file_path}")
else:
    print("File not found. Check your relative pathing!")

In [ ]:
import time

# Configuration
TRAIN_FEATURES_PATH = Path("../../data/raw/landsat_all_features_training.csv")
BATCH_SIZE = 200
MAX_RETRIES = 3
RETRY_DELAY = 10  # seconds

def process_and_save_batches(df, start_batch=0):
    total_rows = len(df)
    # Calculate the starting row index based on the batch number
    start_row = start_batch * BATCH_SIZE
    
    print(f"Starting processing from Batch {start_batch} (Row {start_row})")

    for i in range(start_row, total_rows, BATCH_SIZE):
        batch_idx = i // BATCH_SIZE
        batch = df.iloc[i : i + BATCH_SIZE]
        
        print(f"\nProcessing batch {batch_idx + 1} (Rows {i} to {i+len(batch)})...")
        
        success = False
        retries = 0
        
        while not success and retries < MAX_RETRIES:
            try:
                # Apply feature extraction
                landsat_batch = batch.progress_apply(compute_Landsat_values, axis=1)
                
                # Check if file exists to determine if we need a header
                # If we are at row 0 AND the file doesn't exist, write header.
                # Otherwise, always append without header.
                file_exists = TRAIN_FEATURES_PATH.exists()
                write_header = not file_exists
                
                landsat_batch.to_csv(
                    TRAIN_FEATURES_PATH, 
                    mode='a', 
                    index=False, 
                    header=write_header
                )
                
                success = True
                
            except Exception as e:
                retries += 1
                print(f"Error in batch: {e}. Retrying {retries}/{MAX_RETRIES} in {RETRY_DELAY}s...")
                time.sleep(RETRY_DELAY)
        
        if not success:
            print(f"!! Failed to process batch {batch_idx + 1} after {MAX_RETRIES} attempts. Skipping...")

# WARNING: SKIPPED BATCH 26

In [ ]:
process_and_save_batches(Water_Quality_df, 1)

In [ ]:
landsat_train_features_all = pd.read_csv(TRAIN_FEATURES_PATH)

landsat_train_features_all.shape

### Missing batches: [26, 30, 33, 37, 38, 40, 41, 42, 43, 44, 45, 46]

In [ ]:
# Your specific batch list
SKIPPED_BATCHES = [26, 30, 33, 37, 38, 40, 41, 42, 43, 44, 45, 46]
BATCH_SIZE = 200
TOTAL_ROWS = 9319

recovered_data = {}

def get_start_row(batch_num):
    """
    Adjusted logic: If Batch 26 starts at 5000, 
    then Batch 1 starts at 0.
    """
    return (batch_num - 1) * BATCH_SIZE

def recover_batches(original_df):
    for b_num in SKIPPED_BATCHES:
        start = get_start_row(b_num)
        # Slicing the original dataframe at the EXACT row positions
        batch = original_df.iloc[start : start + BATCH_SIZE]
        
        print(f"Processing Batch {b_num} (Rows {start} to {start + len(batch)})...")
        
        success = False
        while not success:
            try:
                # Apply your extraction function
                res = batch.progress_apply(compute_Landsat_values, axis=1)
                recovered_data[b_num] = res
                success = True
            except Exception as e:
                print(f"Error in Batch {b_num}: {e}. Retrying...")
                time.sleep(10)

In [ ]:
# Run the recovery (Ensure 'df' is your original source coordinates dataframe)
recover_batches(Water_Quality_df)

In [ ]:
def assemble_perfect_file(gappy_csv_path, recovered_dict, final_path):
    df_existing = pd.read_csv(gappy_csv_path)
    
    final_list = []
    csv_pointer = 0
    
    # We need 47 batches to reach 9319 rows
    for b_num in range(1, 48):
        if b_num in recovered_dict:
            # Drop in the recovered API data
            final_list.append(recovered_dict[b_num])
            print(f"Slot {b_num}: Filled with RECOVERED data.")
        else:
            # Pull the next available chunk from your current CSV
            # We calculate how many rows SHOULD be in this batch
            start = get_start_row(b_num)
            expected_len = min(BATCH_SIZE, TOTAL_ROWS - start)
            
            chunk = df_existing.iloc[csv_pointer : csv_pointer + expected_len]
            final_list.append(chunk)
            
            # Move the pointer forward by the number of rows we actually took
            csv_pointer += len(chunk)
            print(f"Slot {b_num}: Filled with {len(chunk)} rows from EXISTING CSV.")

    # Combine and Save
    full_df = pd.concat(final_list, ignore_index=True)
    full_df.to_csv(final_path, index=False)
    
    print(f"\nSuccess! Final row count: {len(full_df)} / {TOTAL_ROWS}")

# Run the final assembly
assemble_perfect_file(TRAIN_FEATURES_PATH, recovered_data, "../../data/raw/landsat_final_aligned.csv")

In [ ]:
final = pd.read_csv("../../data/raw/landsat_final_aligned.csv")

final.head()

In [ ]:
# Upload from your project folder to the Snowflake stage
session.sql(f"""
    PUT file://../../data/raw/landsat_final_aligned.csv
    '@USER$.PUBLIC."EY-AI-and-Data-Challenge"/versions/live/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("File uploaded to Snowflake successfully!")

**NDMI and MNDWI Indices**

In this notebook, we compute two commonly used water-related indices from the extracted Landsat bands:

- **NDMI (Normalized Difference Moisture Index):**  
  Measures vegetation water content and surface moisture.  
  Computed as *(NIR - SWIR16) / (NIR + SWIR16)*.

- **MNDWI (Modified Normalized Difference Water Index):**  
  Highlights open water features by enhancing water reflectance and suppressing built-up areas.  
  Computed as *(Green - SWIR16) / (Green + SWIR16)*.

An **epsilon value** (*eps = 1e-10*) is added to the denominators to avoid division by zero.  
These indices are widely used in hydrological and water quality analyses for detecting water presence and vegetation moisture levels.


In [7]:
# Create indices: NDMI and MNDWI
eps = 1e-10
landsat_train_features['NDMI'] = (landsat_train_features['nir'] - landsat_train_features['swir16']) / (landsat_train_features['nir'] + landsat_train_features['swir16'] + eps)
landsat_train_features['MNDWI'] = (landsat_train_features['green'] - landsat_train_features['swir16']) / (landsat_train_features['green'] + landsat_train_features['swir16'] + eps)

In [8]:
landsat_train_features['Latitude'] = Water_Quality_df['Latitude']
landsat_train_features['Longitude'] = Water_Quality_df['Longitude']
landsat_train_features['Sample Date'] = Water_Quality_df['Sample Date']
landsat_train_features = landsat_train_features[['Latitude', 'Longitude', 'Sample Date', 'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI']]

In [ ]:
# Preview File
landsat_train_features.head()

In [ ]:
landsat_train_features.to_csv("/tmp/landsat_features_training_200.csv",index = False)

In [ ]:
session.sql("""
    PUT file:///tmp/landsat_features_training_200.csv
    'snow://workspace/USER$.PUBLIC."EY-AI-and-Data-Challenge"/versions/live/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("File saved! Refresh the browser to see the files in the sidebar")

**Note:** If you're using your own workspace, remember to replace "EY-AI-and-Data-Challenge" with your workspace name in the file path.

### Extracting features for the validation dataset

In [11]:
Validation_df=pd.read_csv('submission_template.csv')
display(Validation_df.head())

In [12]:
Validation_df.shape

In [ ]:
# Extract band values from Landsat for submission dataset
val_features_path = "landsat_features_validation.csv"

print("🚀 Running Landsat feature extraction for validation data...")
landsat_val_features = Validation_df.progress_apply(compute_Landsat_values, axis=1)
landsat_val_features.to_csv(val_features_path, index=False)

In [14]:
# Create indices: NDMI and MNDWI
eps = 1e-10
landsat_val_features['NDMI'] = (landsat_val_features['nir'] - landsat_val_features['swir16']) / (landsat_val_features['nir'] + landsat_val_features['swir16'])
landsat_val_features['MNDWI'] = (landsat_val_features['green'] - landsat_val_features['swir16']) / (landsat_val_features['green'] + landsat_val_features['swir16'] + eps)

In [15]:
landsat_val_features['Latitude'] = Validation_df['Latitude']
landsat_val_features['Longitude'] = Validation_df['Longitude']
landsat_val_features['Sample Date'] = Validation_df['Sample Date']
landsat_val_features = landsat_val_features[['Latitude', 'Longitude', 'Sample Date', 'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI']]

In [ ]:
# Preview File
landsat_val_features.head()

In [ ]:
landsat_val_features.to_csv("/tmp/landsat_features_validation.csv",index = False)

In [ ]:
session.sql("""
    PUT file:///tmp/landsat_features_validation.csv
    'snow://workspace/USER$.PUBLIC."EY-AI-and-Data-Challenge"/versions/live/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("File saved! Refresh the browser to see the files in the sidebar")

**Note:** If you're using your own workspace, remember to replace "EY-AI-and-Data-Challenge" with your workspace name in the file path.